# Tasks, distributional models, and ensembles

This notebook turns NAMpy's installed registries into executable
capability reports. It discovers task variants at runtime, fits each
supported neural architecture under a tiny common budget, evaluates the
distribution-family catalog, and demonstrates independent ensembles.
A failed fit is recorded rather than silently converted into a checkmark.


## 1. Registry-driven experimental setup

Architecture support and successful demonstration are different facts.
The registry states what an architecture exposes; the sweep records what
executed in this environment. Fast mode uses one development batch per
estimator so broad interface coverage remains practical. Disable fast
mode for meaningful learning curves and comparisons.


In [1]:
from pathlib import Path
import tempfile
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#F97316", "#10B981", "#8B5CF6", "#EF4444"]

FAST_MODE = True
N_SAMPLES = 160 if FAST_MODE else 5000
MAX_EPOCHS = 2 if FAST_MODE else 150
MAX_STEPS = 2 if FAST_MODE else -1
RANDOM_STATE = 7
CHECKPOINT_ROOT = Path(tempfile.mkdtemp(prefix="nampy-notebook-"))

NEURAL_FIT_KWARGS = {
    "max_epochs": MAX_EPOCHS,
    "max_steps": MAX_STEPS,
    "batch_size": 64 if FAST_MODE else 256,
    "random_state": RANDOM_STATE,
    "checkpoint_path": CHECKPOINT_ROOT,
    "logger": False,
    "enable_progress_bar": False,
    "enable_model_summary": False,
    "num_sanity_val_steps": 0,
}

rng = np.random.default_rng(RANDOM_STATE)

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from nampy import models
from nampy.neural.registry import architectures

n = 48 if FAST_MODE else N_SAMPLES
x = np.linspace(0.03, 0.97, n)
groups = pd.Categorical(np.resize(np.array(["a", "b", "c"]), n))
X = pd.DataFrame({"x1": x, "x2": np.cos(2 * np.pi * x), "group": groups})
y_reg = np.sin(2 * np.pi * x) + 0.25 * x + 0.2 * (groups == "b")
y_cls = np.resize(np.array([0, 1]), n)
y_lss = y_reg + np.exp(-1.2 + 0.8 * x) * rng.normal(size=n)

SWEEP_FIT_KWARGS = {
    **NEURAL_FIT_KWARGS,
    "batch_size": 32,
    "shuffle": False,
    "fast_dev_run": FAST_MODE,
}

MODEL_KWARGS = {
    "linreg": {},
    "nam": {"layer_sizes": [8], "dropout": 0.0},
    "sian": {"layer_sizes": [8], "interactions": [("x1", "x2")]},
    "snam": {"layer_sizes": [8], "dropout": 0.0, "group_lasso_lambda": 0.01},
    "gpnam": {"rff_num_feat": 8},
    "igann": {"n_hid": 4, "n_estimators": 3, "early_stopping": 2},
    "nbm": {"layer_sizes": [8], "num_bases": 4, "dropout": 0.0, "bases_dropout": 0.0, "output_penalty": 0.01},
    "nbm_spam": {"layer_sizes": [8], "num_bases": 4, "ranks": [4], "batch_norm": False},
    "spam": {"ranks": [4], "dropout": 0.0},
    "natt": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "namformer": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "treenam": {"tree_depth": 2, "tree_lamda": 0.01},
    "ensemble_treenam": {"tree_depth": 2, "tree_lamda": 0.01, "num_estimators": 2},
    "nodegam": {"num_trees": 4, "num_layers": 1, "depth": 2, "last_dropout": 0.0, "anneal_steps": 10, "interaction_degree": 1},
    "qnam": {"layer_sizes": [8], "dropout": 0.0, "distributional_kwargs": {"quantiles": [0.1, 0.5, 0.9]}},
    "spline_nam": {"n_knots": 5, "smoothing": 0.01},
}

def estimator_class(spec, suffix):
    return getattr(models, f"{spec.estimator_prefix}{suffix}", None)

def make_estimator(name, spec, suffix, **extra):
    cls = estimator_class(spec, suffix)
    if cls is None:
        return None
    kwargs = dict(MODEL_KWARGS[name])
    kwargs.update(extra)
    return cls(**kwargs)

def reconstruction_error(estimator, features):
    components = estimator.predict_components(features, center=True)
    link = np.asarray(components.link)
    def aligned(value):
        array = np.asarray(value, dtype=float)
        if link.ndim == 1 and array.shape == (link.shape[0], 1):
            return array[:, 0]
        if link.ndim == 2 and array.shape == (link.shape[0],):
            return array[:, None]
        return array
    reconstruction = aligned(components.intercept)
    for contribution in components.terms.values():
        reconstruction = reconstruction + aligned(contribution)
    if components.offset is not None:
        reconstruction = reconstruction + aligned(components.offset)
    error = float(np.max(np.abs(reconstruction - link)))
    try:
        components.validate_additive_reconstruction(rtol=1e-5, atol=1e-5)
        valid = True
    except ValueError:
        valid = False
    return error, valid


## 2. Regression across architectures

Every registered regression architecture is fitted on the same rows. The
score, runtime, additive reconstruction, and term importance are recorded.
Architecture-specific output is added where available. This confirms the
installed runtime rather than relying on a copied documentation matrix.


In [2]:
regression_models = {}
regression_rows = []
for name, spec in architectures().items():
    if not spec.supports("regression"):
        continue
    estimator = make_estimator(name, spec, "Regressor")
    started = time.perf_counter()
    try:
        estimator.fit(X, y_reg, **SWEEP_FIT_KWARGS)
        importance = estimator.term_importance(X, center=True)
        error, reconstruction_valid = reconstruction_error(estimator, X)
        row = {
            "architecture": name,
            "status": "Demonstrated",
            "R2": estimator.score(X, y_reg),
            "seconds": time.perf_counter() - started,
            "reconstruction error": error,
            "reconstruction valid": reconstruction_valid,
            "most important term": importance.iloc[0]["term"] if len(importance) else None,
        }
        if hasattr(estimator, "local_term_importance") and spec.supports("local_term_importance"):
            row["special output"] = "local importance"
        elif name == "igann":
            row["special output"] = f"{estimator.n_estimators_} boosting stages"
        else:
            row["special output"] = None
        regression_models[name] = estimator
    except Exception as error:
        row = {"architecture": name, "status": f"Failed: {type(error).__name__}", "error": str(error)}
    regression_rows.append(row)

regression_report = pd.DataFrame(regression_rows)
regression_report


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

,architecture,status,R2,seconds,reconstruction error,reconstruction valid,most important term,special output
0,linreg,Demonstrated,-0.655687,0.689414,5.790207e-08,True,group,None
1,nam,Demonstrated,-0.935797,0.588733,3.894093e-08,True,x1,None
2,sian,Demonstrated,-0.013892,0.508215,4.004687e-08,True,x1:x2,None
3,snam,Demonstrated,-0.508757,0.619273,4.639878e-08,True,group,None
4,gpnam,Demonstrated,0.033354,0.227299,2.759043e-08,True,x1,None
5,igann,Demonstrated,0.081993,0.437595,7.450581e-09,True,x1,3 boosting stages
6,nbm,Demonstrated,-0.111538,1.315372,7.450581e-09,True,group[1],None
7,nbm_spam,Demonstrated,-0.746124,0.697176,4.726462e-08,True,x1,None
8,spam,Demonstrated,-0.552149,0.524977,3.259629e-08,True,x1,local importance
9,natt,Demonstrated,-0.378422,0.924353,4.639151e-08,True,group,None


## 3. Classification across architectures

Classifier components reconstruct logits (binary) or class-score vectors
before the probability transform. Log loss evaluates probability quality,
ROC-AUC evaluates ranking for the positive class, accuracy evaluates hard
labels, and calibration error compares binned predicted and empirical
frequencies. Link-scale terms explain the model score—not an additive
decomposition of probability.


In [3]:
def expected_calibration_error(y_true, probability, bins=6):
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(probability, edges) - 1, 0, bins - 1)
    total = len(y_true)
    return sum(
        np.sum(index == b) / total * abs(np.mean(probability[index == b]) - np.mean(y_true[index == b]))
        for b in range(bins) if np.any(index == b)
    )

classification_models = {}
classification_rows = []
for name, spec in architectures().items():
    if not spec.supports("classification"):
        continue
    estimator = make_estimator(name, spec, "Classifier")
    started = time.perf_counter()
    try:
        estimator.fit(X, y_cls, **SWEEP_FIT_KWARGS)
        probability = estimator.predict_proba(X)[:, 1]
        label = estimator.predict(X)
        importance = estimator.term_importance(X, center=True)
        error, reconstruction_valid = reconstruction_error(estimator, X)
        classification_rows.append({
            "architecture": name, "status": "Demonstrated",
            "log loss": log_loss(y_cls, probability),
            "ROC-AUC": roc_auc_score(y_cls, probability),
            "accuracy": accuracy_score(y_cls, label),
            "calibration error": expected_calibration_error(y_cls, probability),
            "reconstruction error": error,
            "reconstruction valid": reconstruction_valid,
            "most important term": importance.iloc[0]["term"] if len(importance) else None,
            "seconds": time.perf_counter() - started,
        })
        classification_models[name] = estimator
    except Exception as error:
        classification_rows.append({"architecture": name, "status": f"Failed: {type(error).__name__}", "error": str(error)})

classification_report = pd.DataFrame(classification_rows)
classification_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_acc', ..., logger=True)` but have no logge

/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_auroc', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`


...

TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/projects/package/NAMpy/nampy/neural/architectures/components/additive_trees.py:133: UserWarning: Data-aware initialization is performed on less than 1000 data points. This may cause instability. To avoid potential problems, run this model on a data batch with at least 1000 data samples. You can do so manually before training. Use with torch.no_grad() for memory efficienc

/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_acc', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_auroc', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_precision', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
`Trainer.fit` stopped: `max_steps=1` reached.


,architecture,status,log loss,ROC-AUC,accuracy,calibration error,reconstruction error,reconstruction valid,most important term,seconds
0,linreg,Demonstrated,0.737526,0.506944,0.458333,0.157360,9.110227e-08,True,group,0.701402
1,nam,Demonstrated,0.708630,0.517361,0.500000,0.086789,3.895548e-08,True,x1,0.548677
2,sian,Demonstrated,0.694002,0.487847,0.520833,0.004097,2.328306e-08,True,x1:x2,0.467867
3,snam,Demonstrated,0.705542,0.506944,0.500000,0.069636,3.899186e-08,True,group,0.611657
4,gpnam,Demonstrated,0.693080,0.461806,0.562500,0.054147,3.725290e-09,True,group[2],0.501542
5,igann,Demonstrated,0.693147,0.500000,0.500000,0.000000,0.000000e+00,True,x1,0.261684
6,nbm,Demonstrated,0.693212,0.501736,0.500000,0.005203,6.519258e-09,True,group[1],0.508273
7,nbm_spam,Demonstrated,0.712722,0.491319,0.500000,0.097695,4.330650e-08,True,x1,0.527359
8,spam,Demonstrated,0.694389,0.512153,0.520833,0.020330,2.048910e-08,True,x1,0.432725
9,natt,Demonstrated,0.705269,0.491319,0.520833,0.039987,4.642061e-08,True,group,0.668536


## 4. Location-scale regression across architectures

Normal LSS models predict a location and positive scale. Each parameter
has its own link-scale additive predictor, so raw component reconstruction
is checked column by column. Mean negative log likelihood evaluates the
complete conditional distribution; interval coverage checks whether the
predicted scale is calibrated. QNAM is intentionally handled in the
quantile gallery because its fixed family is not normal.


In [4]:
lss_models = {}
lss_rows = []
for name, spec in architectures().items():
    if not spec.supports("distributional") or name == "qnam":
        continue
    estimator = make_estimator(name, spec, "LSS", family="normal")
    started = time.perf_counter()
    try:
        estimator.fit(X, y_lss, **SWEEP_FIT_KWARGS)
        parameters = estimator.predict(X)
        raw = estimator.predict(X, raw=True)
        lower = parameters[:, 0] - 1.645 * parameters[:, 1]
        upper = parameters[:, 0] + 1.645 * parameters[:, 1]
        error, reconstruction_valid = reconstruction_error(estimator, X)
        lss_rows.append({
            "architecture": name, "status": "Demonstrated",
            "NLL": estimator.evaluate(X, y_lss)["NLL"],
            "minimum scale": parameters[:, 1].min(),
            "90% coverage": np.mean((y_lss >= lower) & (y_lss <= upper)),
            "raw width": raw.shape[1],
            "reconstruction error": error,
            "reconstruction valid": reconstruction_valid,
            "seconds": time.perf_counter() - started,
        })
        lss_models[name] = estimator
    except Exception as error:
        lss_rows.append({"architecture": name, "status": f"Failed: {type(error).__name__}", "error": str(error)})

lss_report = pd.DataFrame(lss_rows)
lss_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/projects/package/NAMpy/nampy/neural/architectures/components/additive_trees.py:133: UserWarning: Data-aware initialization is performed on less than 1000 data points. This may cause instability. To avoid potential problems, run this model on a data batch with at least 1000 data samples. You can do so manually before training. Use with torch.no_grad() for memory efficienc

,architecture,status,NLL,minimum scale,90% coverage,raw width,reconstruction error,reconstruction valid,seconds
0,linreg,Demonstrated,1.450104,0.464850,0.833333,2,7.619383e-08,True,6.270807
1,nam,Demonstrated,1.414365,0.489886,0.729167,2,4.300819e-08,True,0.817242
2,sian,Demonstrated,1.178725,0.730420,0.979167,2,4.610047e-08,True,0.505841
3,snam,Demonstrated,1.312788,0.760605,0.875000,2,5.384936e-08,True,2.278448
4,gpnam,Demonstrated,1.187974,0.700404,0.895833,2,6.519258e-09,True,1.360483
5,igann,Demonstrated,1.195254,0.710836,0.916667,2,2.235174e-08,True,0.968030
6,nbm,Demonstrated,1.316261,0.694993,0.770833,2,2.328306e-08,True,0.934946
7,nbm_spam,Demonstrated,1.469806,0.525279,0.687500,2,3.320724e-08,True,1.014350
8,spam,Demonstrated,1.541036,0.521381,0.708333,2,6.332994e-08,True,0.588561
9,natt,Demonstrated,1.665266,0.468499,0.833333,2,5.384209e-08,True,0.678925


## 5. Neural distribution-family gallery

A distribution family owns target support, output width, parameter
transforms, likelihood, point prediction, and default metrics. Continuous,
count, discrete/ordered, multivariate, and quantile families therefore
require different targets. The loop fits one small canonical NAMLSS model
per installed family. These one-batch fits test interface and domains;
they are not estimates worth interpreting scientifically.


In [5]:
from nampy.models import NAMLSS
from nampy.neural.distributions.registry import FAMILY_REGISTRY

positive = np.exp(0.2 + 0.4 * x + rng.normal(0, 0.15, n))
counts = rng.poisson(np.exp(0.2 + 0.5 * x))
zero_counts = counts.copy(); zero_counts[::3] = 0
classes3 = np.resize(np.arange(3), n)
composition = rng.dirichlet([2.0, 3.0, 4.0], size=n)
multi_y = np.column_stack([y_lss, np.cos(np.pi * x) + rng.normal(0, 0.1, n)])

family_targets = {
    "normal": y_lss, "poisson": counts, "gamma": positive,
    "beta": np.clip(1 / (1 + np.exp(-y_lss)), 1e-4, 1-1e-4),
    "dirichlet": composition, "studentt": y_lss,
    "negativebinom": counts, "inversegamma": positive,
    "categorical": classes3, "quantile": y_lss, "robustnormal": y_lss,
    "lognormal": positive, "weibull": positive, "loglogistic": positive,
    "zip": zero_counts, "zinb": zero_counts,
    "hurdlepoisson": zero_counts, "hurdlenegativebinom": zero_counts,
    "tweedie": zero_counts.astype(float), "ordinal": classes3,
    "mvnormdiag": multi_y,
}
family_groups = {
    "continuous": {"normal", "gamma", "beta", "studentt", "inversegamma", "robustnormal", "lognormal", "weibull", "loglogistic", "tweedie"},
    "counts": {"poisson", "negativebinom", "zip", "zinb", "hurdlepoisson", "hurdlenegativebinom"},
    "discrete and ordered": {"categorical", "ordinal"},
    "multivariate": {"dirichlet", "mvnormdiag"},
    "quantiles": {"quantile"},
}

def group_for_family(name):
    return next(group for group, names in family_groups.items() if name in names)

family_models = {}
family_rows = []
for name in FAMILY_REGISTRY:
    kwargs = {"layer_sizes": [8], "dropout": 0.0}
    if name == "quantile":
        kwargs["distributional_kwargs"] = {"quantiles": [0.1, 0.5, 0.9]}
    estimator = NAMLSS(family=name, **kwargs)
    target = family_targets[name]
    started = time.perf_counter()
    try:
        estimator.fit(X, target, **SWEEP_FIT_KWARGS)
        raw = estimator.predict(X, raw=True)
        natural = estimator.predict(X)
        metrics = estimator.evaluate(X, target)
        family_rows.append({
            "group": group_for_family(name), "family": name,
            "status": "Demonstrated", "target shape": np.asarray(target).shape,
            "raw parameters": raw.shape[1], "finite output": bool(np.isfinite(natural).all()),
            "metric": next(iter(metrics)), "metric value": next(iter(metrics.values())),
            "seconds": time.perf_counter() - started,
        })
        family_models[name] = estimator
    except Exception as error:
        family_rows.append({
            "group": group_for_family(name), "family": name,
            "status": f"Failed: {type(error).__name__}", "error": str(error),
        })

family_report = pd.DataFrame(family_rows).sort_values(["group", "family"])
family_report


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

,group,family,status,target shape,raw parameters,finite output,metric,metric value,seconds
3,continuous,beta,Demonstrated,"(48,)",2,True,NLL,0.223862,0.482138
2,continuous,gamma,Demonstrated,"(48,)",2,True,NLL,1.656360,0.452622
7,continuous,inversegamma,Demonstrated,"(48,)",2,True,NLL,1.706410,0.502956
13,continuous,loglogistic,Demonstrated,"(48,)",2,True,NLL,2.306082,0.508049
11,continuous,lognormal,Demonstrated,"(48,)",2,True,NLL,1.079826,0.500745
0,continuous,normal,Demonstrated,"(48,)",2,True,NLL,1.414365,0.435955
10,continuous,robustnormal,Demonstrated,"(48,)",2,True,NLL,0.436677,0.489651
5,continuous,studentt,Demonstrated,"(48,)",3,True,NLL,1.725316,0.599168
18,continuous,tweedie,Demonstrated,"(48,)",2,True,NLL,2.722834,0.511965
12,continuous,weibull,Demonstrated,"(48,)",2,True,NLL,2.002926,0.392511


### 5.1 Domain and non-crossing checks

Natural transforms enforce positive rates/scales and unit-interval
probabilities. They do not guarantee calibration. QNAM additionally
enforces ordered quantiles numerically.


In [6]:
domain_checks = {
    "normal scale positive": bool((family_models["normal"].predict(X)[:, 1] > 0).all()),
    "beta parameters positive": bool((family_models["beta"].predict(X) > 0).all()),
    "ZIP finite": bool(np.isfinite(family_models["zip"].predict(X)).all()),
}

qnam = make_estimator("qnam", architectures()["qnam"], "LSS")
qnam.fit(X, y_lss, **SWEEP_FIT_KWARGS)
quantiles = qnam.predict(X)
assert np.all(np.diff(quantiles, axis=1) >= -1e-7)
domain_checks["QNAM non-crossing"] = True
domain_checks


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

{'normal scale positive': True,
 'beta parameters positive': True,
 'ZIP finite': True,
 'QNAM non-crossing': True}

## 6. Independent neural ensembles

`NeuralEnsemble` clones and fits regressors or classifiers independently.
Bootstrapping changes member datasets; random seeds change initialization
and data order; `n_jobs` controls parallel fitting. Predictions are
averaged on the response scale, while terms and links are averaged on the
additive scale. Persistence is inherited from the member estimators only;
the complete ensemble can be serialized with joblib.


In [7]:
from joblib import dump, load
from nampy.models import NAMRegressor, NeuralEnsemble

ensemble = NeuralEnsemble(
    NAMRegressor(layer_sizes=[8], dropout=0.0),
    n_estimators=3, random_state=RANDOM_STATE, n_jobs=1, bootstrap=True,
)
ensemble.fit(X, y_reg, **SWEEP_FIT_KWARGS)
ensemble_prediction = ensemble.predict(X)
ensemble_components = ensemble.predict_components(X, center=True)
uncertainty = ensemble.predict_component_uncertainty(X, center=True)
ensemble_components.validate_additive_reconstruction(rtol=1e-5, atol=1e-5)

ensemble_path = CHECKPOINT_ROOT / "neural-ensemble.joblib"
dump(ensemble, ensemble_path)
restored_ensemble = load(ensemble_path)
np.testing.assert_allclose(restored_ensemble.predict(X), ensemble_prediction)

display({
    "members": uncertainty.n_estimators,
    "mean response SD": float(np.mean(uncertainty.response_std)),
    "mean term SD": {name: float(values.mean()) for name, values in uncertainty.term_std.items()},
    "persistence equality": True,
})


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

GPU available: False, used: False


...

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.


/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/home/ad32/.local/lib/python3.10/site-packages/lightning/pytorch/core/module.py:522: You called `self.log('val_loss', ..., logger=True)` but have no logger

{'members': 3,
 'mean response SD': 0.2144484966993332,
 'mean term SD': {'x1': 0.040574980867280654,
  'x2': 0.12966108847444438,
  'group': 0.03326916999695132},
 'persistence equality': True}

### 6.1 Three different meanings of uncertainty

- **Ensemble disagreement** measures variation among fitted neural members
  under seeds and/or bootstrap samples. It is not automatically a
  calibrated confidence interval.
- **GAM standard errors** arise from an approximate sampling covariance for
  fitted coefficients, conditional on model assumptions and optionally
  including smoothing uncertainty.
- **LSS conditional uncertainty** is the predicted response distribution
  for a new outcome given its features. It can be large even if parameter
  estimates are known precisely.


## 7. Final capability report

Cells use four states. **Demonstrated** means this notebook executed the
task successfully. **Supported but not demonstrated** comes from the
registry but lacked a successful run. **Unavailable** means the registry
does not expose the capability. **Not applicable** is reserved for a
concept that does not make sense for that architecture.


In [8]:
def task_status(name, capability, report):
    spec = architectures()[name]
    if not spec.supports(capability):
        return "Unavailable"
    match = report.loc[report["architecture"] == name]
    if len(match) and match.iloc[0]["status"] == "Demonstrated":
        return "Demonstrated"
    return "Supported but not demonstrated"

capability_rows = []
for name, spec in architectures().items():
    capability_rows.append({
        "model": name,
        "Regression": task_status(name, "regression", regression_report),
        "Classification": task_status(name, "classification", classification_report),
        "LSS": "Demonstrated" if name == "qnam" else task_status(name, "distributional", lss_report),
        "Main effects": "Demonstrated" if spec.supports("additive_components") else "Unavailable",
        "Pairwise interactions": "Supported but not demonstrated" if spec.supports("interactions") else "Unavailable",
        "Higher-order interactions": "Supported but not demonstrated" if name in {"sian", "nbm", "spam", "nbm_spam"} else "Not applicable",
        "Sparsity": "Supported but not demonstrated" if name in {"snam", "sian", "nbm", "igann"} else "Not applicable",
        "Local importance": "Demonstrated" if name == "spam" and name in regression_models else ("Unavailable" if not spec.supports("local_term_importance") else "Supported but not demonstrated"),
        "Quantiles": "Demonstrated" if name == "qnam" else "Unavailable",
    })

final_capability_report = pd.DataFrame(capability_rows)
final_capability_report


,model,Regression,Classification,LSS,Main effects,Pairwise interactions,Higher-order interactions,Sparsity,Local importance,Quantiles
0,linreg,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Unavailable,Not applicable,Not applicable,Unavailable,Unavailable
1,nam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Not applicable,Unavailable,Unavailable
2,sian,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Supported but not demonstrated,Unavailable,Unavailable
3,snam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Supported but not demonstrated,Unavailable,Unavailable
4,gpnam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Not applicable,Unavailable,Unavailable
5,igann,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Unavailable,Not applicable,Supported but not demonstrated,Unavailable,Unavailable
6,nbm,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Supported but not demonstrated,Unavailable,Unavailable
7,nbm_spam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Not applicable,Unavailable,Unavailable
8,spam,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Supported but not demonstrated,Not applicable,Demonstrated,Unavailable
9,natt,Demonstrated,Demonstrated,Demonstrated,Demonstrated,Supported but not demonstrated,Not applicable,Not applicable,Unavailable,Unavailable


## 8. Reading the reports responsibly

A successful one-batch fit proves API wiring, output domains, and
reconstruction—not predictive quality. A registry capability proves an
exposed contract—not that the architecture is the right scientific model.
For analysis, disable fast mode, choose metrics before fitting, tune inside
cross-validation, use repeated splits, inspect term stability, and compare
against simple baselines.
